# Connect to Tools Gateway (AgentCore path — Module 3b)

::: **Use this notebook if you followed Module 3b.** For the MCP path (Module 3a), use
**notebook 04a** instead. :::

Module 3b deploys `ac-tools-gateway` directly as an AgentCore Gateway with Lambda targets. The
auth story is the same as 04a: FAST's agent needs to authenticate to the gateway using the
shared Cognito pool created by the Registry stack.

Steps mirror 04a, with the difference that:

- The gateway is read from `workshop-agentcore-stack` output `GatewayId`
- The M2M client/secret come from `workshop-cognito-m2m-secret`


## Step 1: Look Up Module 3b Credentials


In [ ]:
import boto3, json, os

REGION = (
    os.environ.get("AWS_REGION")
    or os.environ.get("AWS_DEFAULT_REGION")
    or boto3.session.Session().region_name
)
if not REGION:
    raise RuntimeError(
        "No AWS region configured. Set AWS_REGION (or AWS_DEFAULT_REGION), "
        "or run `aws configure set region <region>`."
    )
cfn = boto3.client("cloudformation", region_name=REGION)
sm = boto3.client("secretsmanager", region_name=REGION)
agentcore = boto3.client("bedrock-agentcore-control", region_name=REGION)
ssm = boto3.client("ssm", region_name=REGION)

m2m_secret = json.loads(sm.get_secret_value(SecretId="workshop-cognito-m2m-secret")["SecretString"])
M2M_CLIENT_ID = m2m_secret["client_id"]
M2M_CLIENT_SECRET = m2m_secret["client_secret"]

exports = {}
for page in cfn.get_paginator("list_exports").paginate():
    for e in page["Exports"]:
        exports[e["Name"]] = e["Value"]

COGNITO_POOL_ID = exports["workshop-CognitoUserPoolId"]
DISCOVERY_URL = f"https://cognito-idp.{REGION}.amazonaws.com/{COGNITO_POOL_ID}/.well-known/openid-configuration"

print(f"M2M Client ID:  {M2M_CLIENT_ID}")
print(f"Discovery URL:  {DISCOVERY_URL}")

## Step 2: Create the OAuth2 Credential Provider (idempotent)


In [ ]:
PROVIDER_NAME = "workshop-tools-gateway-auth"

provider_config = {
    "customOauth2ProviderConfig": {
        "oauthDiscovery": {"discoveryUrl": DISCOVERY_URL},
        "clientId": M2M_CLIENT_ID,
        "clientSecret": M2M_CLIENT_SECRET,
    }
}

try:
    resp = agentcore.create_oauth2_credential_provider(
        name=PROVIDER_NAME,
        credentialProviderVendor="CustomOauth2",
        oauth2ProviderConfigInput=provider_config,
    )
    print(f"Created provider: {resp.get('credentialProviderArn', PROVIDER_NAME)}")
except agentcore.exceptions.ConflictException:
    print(f"Provider '{PROVIDER_NAME}' already exists — OK")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"Provider '{PROVIDER_NAME}' already exists — OK")
    else:
        raise


## Step 3: Look Up `ac-tools-gateway` and Publish SSM


In [ ]:
stack = cfn.describe_stacks(StackName="workshop-agentcore-stack")["Stacks"][0]
outputs = {o["OutputKey"]: o["OutputValue"] for o in stack["Outputs"]}
GATEWAY_ID = outputs["GatewayId"]
GATEWAY_URL = f"https://{GATEWAY_ID}.gateway.bedrock-agentcore.{REGION}.amazonaws.com/mcp"

ssm.put_parameter(Name="/FAST-stack/gateway_url", Value=GATEWAY_URL, Type="String", Overwrite=True)
ssm.put_parameter(Name="/FAST-stack/gateway_credential_provider", Value=PROVIDER_NAME, Type="String", Overwrite=True)

print(f"Gateway URL:          {GATEWAY_URL}")
print(f"Credential provider:  {PROVIDER_NAME}")


## Step 4: Overwrite `tools/gateway.py`


In [ ]:
GATEWAY_PY = r'''"""AgentCore Gateway MCP client with OAuth2 authentication."""

import logging
import os

from bedrock_agentcore.identity.auth import requires_access_token
from mcp.client.streamable_http import streamablehttp_client
from strands.tools.mcp import MCPClient
from utils.ssm import get_ssm_parameter

logger = logging.getLogger(__name__)

_stack_name = os.environ.get("STACK_NAME", "")
try:
    _provider_name = get_ssm_parameter(f"/{_stack_name}/gateway_credential_provider")
    logger.info("[GATEWAY] Using credential provider from SSM: %s", _provider_name)
except Exception:
    _provider_name = os.environ.get("GATEWAY_CREDENTIAL_PROVIDER_NAME", "")
    logger.info("[GATEWAY] Using credential provider from env: %s", _provider_name)


@requires_access_token(provider_name=_provider_name, auth_flow="M2M", scopes=[])
def _fetch_gateway_token(access_token: str) -> str:
    """Fetch OAuth2 token via AgentCore Identity Token Vault."""
    return access_token


def create_gateway_mcp_client() -> MCPClient:
    """Create MCP client for AgentCore Gateway."""
    stack_name = os.environ.get("STACK_NAME")
    if not stack_name:
        raise ValueError("STACK_NAME environment variable is required")
    if not stack_name.replace("-", "").replace("_", "").isalnum():
        raise ValueError("Invalid STACK_NAME format")
    gateway_url = get_ssm_parameter(f"/{stack_name}/gateway_url")
    logger.info("[GATEWAY] URL: %s", gateway_url)
    return MCPClient(
        lambda: streamablehttp_client(
            url=gateway_url,
            headers={"Authorization": f"Bearer {_fetch_gateway_token()}"},
        ),
        prefix="gateway",
    )
'''

import pathlib, ast
FAST_DIR = pathlib.Path("/workshop/fast-agent")
gateway_py = FAST_DIR / "patterns" / "strands-travel-agent" / "tools" / "gateway.py"
gateway_py.write_text(GATEWAY_PY)
ast.parse(GATEWAY_PY)
print(f"Wrote {gateway_py} and parses cleanly")


## Step 5: Widen the Agent IAM Policy


In [ ]:
import re
backend_stack = FAST_DIR / "infra-cdk" / "lib" / "backend-stack.ts"
text = backend_stack.read_text()
wide = "bedrock-agentcore-identity!default/oauth2/*"

# Check for the NARROW runtime-gateway-auth pattern specifically.
# FAST v0.4.1 already contains the wide pattern elsewhere in this file (on the OAuth2
# provider Lambda's role, unrelated to the runtime role), so checking "wide in text"
# would incorrectly short-circuit and skip widening the runtime role's inline policy.
narrow_re = re.compile(r"bedrock-agentcore-identity!default/oauth2/[^\"\x27`]*runtime-gateway-auth\*")
narrow_hits = narrow_re.findall(text)

if not narrow_hits:
    print(f"Already patched: {backend_stack} (no narrow runtime-gateway-auth pattern found)")
else:
    new_text, n = narrow_re.subn(wide, text)
    backend_stack.write_text(new_text)
    print(f"Patched {backend_stack} ({n} narrow pattern(s) replaced)")
    verify = backend_stack.read_text()
    if narrow_re.search(verify):
        raise RuntimeError(
            f"Widening verification failed: narrow runtime-gateway-auth pattern still in {backend_stack}."
        )
    print(f"Verified: {backend_stack} no longer contains the narrow runtime pattern")

## Step 6: Redeploy


In [ ]:
import os, subprocess

travel_py = FAST_DIR / "patterns" / "strands-travel-agent" / "travel_agent.py"
with travel_py.open("a") as f:
    f.write("# Tools Gateway integration (AgentCore path)\n")
print(f"Touched {travel_py}")

os.chdir(FAST_DIR / "infra-cdk")

# Use try/finally so a partial deploy still gets the SSM overrides re-applied.
import boto3
_ssm = boto3.client("ssm", region_name=REGION)
try:
    r = subprocess.run(
        ["npx", "cdk", "deploy", "--require-approval", "never"],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        raise RuntimeError(
            f"cdk deploy failed (exit {r.returncode}).\n"
            f"stdout tail: {r.stdout[-2500:]}\nstderr tail: {r.stderr[-2500:]}"
        )
    print(r.stdout[-600:])
    print("Redeploy complete")
finally:
    _ssm.put_parameter(Name="/FAST-stack/gateway_url", Value=GATEWAY_URL, Type="String", Overwrite=True)
    _ssm.put_parameter(Name="/FAST-stack/gateway_credential_provider", Value=PROVIDER_NAME, Type="String", Overwrite=True)
    print("Re-applied /FAST-stack/gateway_url and /FAST-stack/gateway_credential_provider")

## Verify

Open the Amplify URL and ask:

> Plan a trip from SFO to Tokyo for 2026-09-15 to 2026-09-18, 2 guests

The agent should return real flight and hotel results via `ac-tools-gateway`.
